In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import pynucastro as pyna
import pandas as pd


In [2]:

# ----------------------------
# Tian et al. (2025) constants
# Best-performing formula in the paper: F3
# ----------------------------
ME = 0.511  # m_e c^2 in MeV

# F3 parameters (Table 1 in Tian 2025)
T_A1 = 14.608
T_A2 = 6.164
T_A3 = 0.545
T_A4 = 3.985
T_A5 = 5.882
T_A6 = 3.610
T_A7 = 1.608
T_A8 = 0.498


@dataclass(frozen=True)
class Nuclide:
    Z: int
    N: int
    A: int
    name: str          # e.g. "he3", "ni56"
    mex_mev: float     # mass excess in MeV (atomic)
    is_stable: bool


# ----------------------------
# Helpers
# ----------------------------
def parity_sign(x: int) -> int:
    """(-1)^x: +1 if even, -1 if odd."""
    return 1 if (x % 2 == 0) else -1


def pairing_delta(Z: int, N: int) -> int:
    """δ = (-1)^Z + (-1)^N ∈ {+2,0,-2}."""
    return parity_sign(Z) + parity_sign(N)


def shell_S_tian(Z: int, N: int) -> float:
    """
    Tian (2025) shell correction:
      S(Z,N)=a4 exp(-((Z-20)^2+(N-24)^2)/30)
            +a5 exp(-((Z-40)^2+(N-50)^2)/40)
            +a6 exp(-((Z-56)^2+(N-82)^2)/34)
            +a7 exp(-((Z-82)^2+(N-132)^2)/11)
    """
    return (
        T_A4 * math.exp(-(((Z - 20) ** 2 + (N - 24) ** 2) / 30.0))
        + T_A5 * math.exp(-(((Z - 40) ** 2 + (N - 50) ** 2) / 40.0))
        + T_A6 * math.exp(-(((Z - 56) ** 2 + (N - 82) ** 2) / 34.0))
        + T_A7 * math.exp(-(((Z - 82) ** 2 + (N - 132) ** 2) / 11.0))
    )


def t12_tian_seconds(Z: int, N: int, Qb_mev: float) -> Optional[float]:
    """
    Tian (2025) F3 empirical beta- half-life.
    Returns T1/2 [s], or None if invalid (e.g., log argument <= 0).
    """
    delta = pairing_delta(Z, N)

    # Main log argument: Qβ + m_e c^2 + a3*delta
    arg1 = Qb_mev + ME + T_A3 * delta
    if arg1 <= 0.0:
        return None

    # "Transition strength" proxy term:
    #   X = Z e^{-N/Z} + N e^{-Z/N} + (N-Z)
    # Must be > 0 for log.
    if Z <= 0 or N <= 0:
        return None
    x = (
        float(Z) * math.exp(-float(N) / float(Z))
        + float(N) * math.exp(-float(Z) / float(N))
        + float(N - Z)
    )
    if x <= 0.0:
        return None

    ln_t = (
        T_A1
        - T_A2 * math.log(arg1)
        + shell_S_tian(Z, N)
        - T_A8 * math.log(x)
    )

    # guard overflow/underflow
    if ln_t > 700:
        return float("inf")
    if ln_t < -800:
        return 0.0
    return math.exp(ln_t)


def lambda_from_t12(t12_s: float) -> float:
    """λ = ln2 / T1/2."""
    if not math.isfinite(t12_s) or t12_s <= 0.0:
        return 0.0
    return math.log(2.0) / t12_s


# ----------------------------
# WinVN reader (tolerant)
# ----------------------------

def read_winvn(path: str | Path) -> List[Nuclide]:
    """
    Reads a WinVN-like file where each line contains at least:
      idx  Z  A  N  name  mass_excess  [stable_flag ...]
    Many WinVN variants exist; this parser is whitespace-based and tolerant.
    """
    nucs: List[Nuclide] = []
    nuclei_in_the_reaction=pd.read_csv(r'winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])
    for i in range(len(nuclei_in_the_reaction)):
            nucleus=pyna.nucdata.nucleus.Nucleus(nuclei_in_the_reaction['name'][i])
            is_stable=False
            if nucleus.tau =='stable':
                is_stable=True
            nucs.append(Nuclide(Z=nucleus.Z, N=nucleus.N, A=nucleus.Z+nucleus.N, name=nuclei_in_the_reaction['name'][i], mex_mev=nuclei_in_the_reaction['Mass excess (Mev)'][i], is_stable=is_stable))

    return nucs
    


# ----------------------------
# REACLIB R1 writer (constant rate)
# ----------------------------

def format_r1_block(parent: Nuclide, daughter: Nuclide, q_mev: float, lam: float, source: str = "zhou17") -> str:
    """
    Writes a 3-line REACLIB-R1-like block:
      line1: parent daughter (padded) + source + Q
      line2: a0..a3
      line3: a4..a6
    For constant rate: rate(T9)=exp(a0), set a0=ln(lam), others 0.
    """
    # Protect: lambda must be >0
    if lam <= 0.0:
        a0 = -1.0e99
    else:
        a0 = math.log(lam)


    # line1: keep it whitespace-separated (parsers using split() will work)
    line1 = f"     {parent.name:>5}{daughter.name:>5}"+" "*28+"wc12w    "+f"{q_mev: .5e}"+" "*10
    line2 = f"{a0: .6e}"+" 0.000000e+00 0.000000e+00 0.000000e+00                      "
    line3 = " 0.000000e+00 0.000000e+00 0.000000e+00                                   "

    return line1 + "\n" + line2 + "\n" + line3 + "\n"


def write_tian2025_beta_decay_file(
    winvn_path: str | Path,
    out_path: str | Path,
    *,
    z_min: int = 8,
    skip_stable_parents: bool = True,
) -> None:
    """
    Create a file containing beta- decay rates computed with Tian (2025) F3 for all nuclides in WinVN.

    Requirements:
      - Qβ computed from atomic mass excess in WinVN: Qβ = mex(parent) - mex(daughter)
      - Only decays where daughter (Z+1, N-1) exists and Qβ>0 are written

    Notes:
      - Tian’s fit domain in the paper: Z,N >= 8 and Qβ > 0 (empirical data selection).
      - This writes constant REACLIB-style blocks (a0=ln λ, others=0).
    """
    nucs = read_winvn(winvn_path)
    by_ZN: Dict[Tuple[int, int], Nuclide] = {(nu.Z, nu.N): nu for nu in nucs}

    out_path = Path(out_path)
    n_written = 0
    n_skipped = 0

    with out_path.open("w", encoding="utf-8") as f:
        f.write("1"+" "*73+"\n")
        f.write(" "*74+"\n")
        f.write(" "*74+"\n")


        for parent in nucs:
            if parent.Z < z_min or parent.N < z_min:
                continue
            if skip_stable_parents and parent.is_stable:
                continue

            daughter = by_ZN.get((parent.Z + 1, parent.N - 1))
            if daughter is None:
                continue

            Qb = parent.mex_mev - daughter.mex_mev
            if Qb <= 0.0:
                continue

            t12 = t12_tian_seconds(parent.Z, parent.N, Qb)
            if t12 is None:
                n_skipped += 1
                continue

            lam = lambda_from_t12(t12)
            f.write(format_r1_block(parent, daughter, Qb, lam, source="tian25"))
            n_written += 1

    print(f"Output: {out_path}")
    print(f"Written beta- rates: {n_written}")
    print(f"Skipped (invalid log args etc.): {n_skipped}")


# ----------------------------
# usage
# ----------------------------
if __name__ == "__main__":
    winvn = "Winvn_v2.0"  # set your real path
    out = "Tian_beta_R1"

    write_tian2025_beta_decay_file(
        winvn_path=winvn,
        out_path=out,
        z_min=8,
        skip_stable_parents=True,
    )


Output: Tian_beta_R1
Written beta- rates: 5145
Skipped (invalid log args etc.): 26
